<a href="https://colab.research.google.com/github/mithra-malicious/carisurg_portfolio/blob/main/Tutorial2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Pandas version:", pd.__version__)
print("Numpy version:", np.__version__)

clinical_bounds = {
    'GCS': {'min': 3.0, "max":15.0},
    'SBP': {'min': 60.0, 'max':250.0},
    'DBP': {'min':30.0, 'max': 150.0},
    'MAP': {'min': 40.0, 'max':180.0 },
    'pulse': {'min': 30.0, 'max':220.0},
    'Temp': {'min': 34.0, 'max':43.0},
    'RR': {'min': 8.0, 'max':60.0},
    'Fio2': {'min': 21.0, 'max': 100.0}
}
print(f"Full clinical rules stated.")


Pandas version: 2.2.2
Numpy version: 2.0.2
Full clinical rules stated.


In [17]:
FILE_PATH = '/content/drive/MyDrive/Copy of EmergencyTriageDataset_Reduced_Dirty.csv'

if not os.path.exists(FILE_PATH):
  path(f"ERROR: Cannot find the file! Make sure it is in: {os.getcwd()}")
else:
  df_clean= pd.read_csv(FILE_PATH)
  print(f"Data loaded successfully: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns")

  display(df_clean.dtypes)

Data loaded successfully: 2205 rows x 11 columns


,0
ID,int64
Age,int64
Gender,object
GCS,object
SBP,object
DBP,float64
MAP,float64
pulse,object
Temp,object
RR,float64


In [18]:
gender_map = {'male': 1, '1':1, 'MALE':1, '1.0':1, 'female':0, 'FEMALE':0, '0':0, '0.0':0}

df_clean['Gender_Clean'] = df_clean['Gender'].astype(str).str.lower().str.strip().map(gender_map)
df_clean = df_clean.drop(columns=['Gender']).rename(columns={'Gender_Clean':'Gender'})
print("Part1: Gender column normalised and converted to clean binary indicators.")

print("\nProcessing clinical vital signs...")
for column, bounds in clinical_bounds.items():
  df_clean[column] = pd.to_numeric(df_clean[column], errors='coerce')
  outliers =  ((df_clean[column]< bounds['min']) | (df_clean[column]> bounds['max'])).sum()
  df_clean.loc[(df_clean[column]<bounds['min']) | (df_clean[column]>bounds['max']),column] = np.nan

  clean_median = df_clean[column].median()
  df_clean[column] = df_clean[column].fillna(clean_median)

  print(f"Column '{column}': Fixed {outliers} impossible outliers. Filled blank susing median ({clean_median:.2f})")

print("\n Cleaning pipeline successfully finalised")

Part1: Gender column normalised and converted to clean binary indicators.

Processing clinical vital signs...
Column 'GCS': Fixed 0 impossible outliers. Filled blank susing median (15.00)
Column 'SBP': Fixed 49 impossible outliers. Filled blank susing median (125.00)
Column 'DBP': Fixed 3 impossible outliers. Filled blank susing median (78.00)
Column 'MAP': Fixed 3 impossible outliers. Filled blank susing median (93.33)
Column 'pulse': Fixed 43 impossible outliers. Filled blank susing median (90.00)
Column 'Temp': Fixed 15 impossible outliers. Filled blank susing median (37.00)
Column 'RR': Fixed 0 impossible outliers. Filled blank susing median (18.00)
Column 'Fio2': Fixed 0 impossible outliers. Filled blank susing median (21.00)

 Cleaning pipeline successfully finalised


In [22]:
print("-- Final Dataset Health Veification ---")
print(df_clean.dtypes)

print("\nMissing (NaN) vales left in dataset:")
print(df_clean.isnull().sum())
print("\n---Comprehensice Summary Matrix After Cleaning---")

display(df_clean.describe())
df_clean.head(10)

-- Final Dataset Health Veification ---
ID          int64
Age         int64
GCS       float64
SBP       float64
DBP       float64
MAP       float64
pulse     float64
Temp      float64
RR        float64
Fio2      float64
Gender      int64
dtype: object

Missing (NaN) vales left in dataset:
ID        0
Age       0
GCS       0
SBP       0
DBP       0
MAP       0
pulse     0
Temp      0
RR        0
Fio2      0
Gender    0
dtype: int64

---Comprehensice Summary Matrix After Cleaning---


,ID,Age,GCS,SBP,DBP,MAP,pulse,Temp,RR,Fio2,Gender
count,2205.000000,2205.000000,2205.000000,2205.000000,2205.000000,2205.000000,2205.000000,2205.000000,2205.000000,2205.000000,2205.000000
mean,1154.987755,61.829478,14.425850,126.782766,77.367347,93.849020,94.326984,37.244717,20.239683,24.979592,0.533333
std,677.167364,18.485363,1.375031,26.619274,16.316160,18.680738,19.881720,0.794448,5.718026,10.101438,0.499001
min,1.000000,18.000000,3.000000,60.000000,30.000000,40.670000,40.000000,35.000000,12.000000,21.000000,0.000000
25%,577.000000,50.000000,15.000000,110.000000,70.000000,83.330000,80.000000,37.000000,17.000000,21.000000,0.000000
50%,1135.000000,64.000000,15.000000,125.000000,78.000000,93.330000,90.000000,37.000000,18.000000,21.000000,1.000000
75%,1703.000000,77.000000,15.000000,140.000000,87.000000,103.330000,106.000000,37.400000,21.000000,21.000000,1.000000
max,2384.000000,98.000000,15.000000,250.000000,150.000000,180.000000,170.000000,41.700000,50.000000,100.000000,1.000000


,ID,Age,GCS,SBP,DBP,MAP,pulse,Temp,RR,Fio2,Gender
0,1,34,15.0,93.0,67.0,75.67,128.0,36.8,14.0,21.0,0
1,2,20,15.0,130.0,90.0,103.33,80.0,37.0,16.0,21.0,1
2,3,77,14.0,163.0,105.0,124.33,92.0,36.8,18.0,21.0,0
3,4,23,8.0,100.0,60.0,73.33,100.0,37.0,12.0,100.0,0
4,5,86,15.0,150.0,90.0,110.00,85.0,37.0,19.0,21.0,0
5,6,42,15.0,100.0,60.0,73.33,99.0,37.0,20.0,21.0,1
6,7,75,15.0,120.0,80.0,93.33,99.0,37.0,25.0,21.0,0
7,8,25,15.0,100.0,50.0,66.67,85.0,37.0,25.0,21.0,1
8,9,67,15.0,110.0,70.0,83.33,78.0,37.0,16.0,21.0,0
9,11,82,15.0,153.0,82.0,105.67,130.0,37.0,19.0,21.0,0
